In [1]:
import pandas as pd
df_train = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/RE-TACRED/Train/MINT/RE_TACRED_train_phrase.csv')
print(df_train.shape) 
df_train.head(2)

(1850, 15)


,split,id,text,token,subject,subj_start,subj_end,subj_type,object,obj_start,obj_end,obj_type,relation,gpt4_phrase,gpt4_phrase_preprocessed
0,train,61b3a65fb9acc5c72e88,"* `` Updates , Dcr -LRB- DCR -RRB- '' < Dcr.Up...","['*', '``', 'Updates', ',', 'Dcr', '-LRB-', 'D...",DCR,6,6,ORGANIZATION,Dcr,4,4,ORGANIZATION,org:alternate_names,alias,alias
1,train,61b3a65fb916e324f16c,-LRB- While I 'm sure that Fidelity also reser...,"['-LRB-', 'While', 'I', ""'m"", 'sure', 'that', ...",Fidelity,14,14,ORGANIZATION,Fidelity,6,6,ORGANIZATION,org:alternate_names,self-reference,self-reference


In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize


texts = df_train["gpt4_phrase"].fillna("").astype(str).tolist()



print("num texts:", len(texts))


model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("embedding shape:", embeddings.shape)


def estimate_k_by_zhang2021(embeddings, k0, random_state=42):
    """
    Zhang et al. (2021a) 방식의 unknown K 추정.

    Args:
        embeddings   : clustering할 embedding matrix, shape = (N, dim)
        k0           : over-clustering에 사용할 큰 cluster 개수
        random_state : random seed

    Returns:
        estimated_k  : 추정된 cluster 개수
        labels_k0    : K0로 clustering한 결과
        cluster_info : cluster별 size 정보
    """

    n = len(embeddings)

    kmeans = KMeans(
        n_clusters=k0,
        random_state=random_state,
        n_init="auto"
    )

    labels_k0 = kmeans.fit_predict(embeddings)

    cluster_sizes = pd.Series(labels_k0).value_counts().sort_index()

    threshold = n / k0

    valid_clusters = cluster_sizes[cluster_sizes >= threshold].index.tolist()
    estimated_k = len(valid_clusters)

    cluster_info = pd.DataFrame({
        "cluster_id": cluster_sizes.index,
        "size": cluster_sizes.values,
        "threshold": threshold,
        "is_valid": cluster_sizes.values >= threshold
    })

    return estimated_k, labels_k0, cluster_info

num texts: 1850


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

embedding shape: (1850, 384)


In [3]:
k0 = 300

estimated_k, labels_k0, cluster_info = estimate_k_by_zhang2021(
    embeddings=embeddings,
    k0=k0,
    random_state=42
)

print("Estimated K:", estimated_k)
cluster_info

Estimated K: 79


,cluster_id,size,threshold,is_valid
0,0,10,6.166667,True
1,1,33,6.166667,True
2,2,11,6.166667,True
3,3,20,6.166667,True
4,4,21,6.166667,True
...,...,...,...,...
295,295,1,6.166667,False
296,296,1,6.166667,False
297,297,3,6.166667,False
298,298,3,6.166667,False


In [4]:
kmeans = KMeans(
    n_clusters=estimated_k,
    random_state=42,
    n_init="auto"
)

df_train["cluster_unknown_k"] = kmeans.fit_predict(embeddings)


# Evaluation

In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_rand_score,
    v_measure_score,
    homogeneity_score,
    completeness_score,
    confusion_matrix
)
from scipy.optimize import linear_sum_assignment


def evaluate_clustering(df: pd.DataFrame, pred_col: str, gold_col: str = "relation"):

    
    non_na_df = df[~df[pred_col].isna()].copy()

    if len(non_na_df) == 0:
        raise ValueError(f"{pred_col}에 유효한 값이 없습니다.")

    y_true = non_na_df[gold_col].astype(str).values
    y_pred = non_na_df[pred_col].astype(str).values

    
    overall_purity = (
        non_na_df
        .groupby(pred_col)[gold_col]
        .value_counts()
        .groupby(level=0)
        .apply(lambda x: x.max())
        .sum()
        / len(non_na_df)
    )

    
    ari = adjusted_rand_score(y_true, y_pred)

    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v_f1 = v_measure_score(y_true, y_pred)

    
    def b3_precision_recall_f1(y_true, y_pred):

        true_groups = {}
        pred_groups = {}

        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            true_groups.setdefault(t, set()).add(i)
            pred_groups.setdefault(p, set()).add(i)

        P_sum, R_sum = 0.0, 0.0
        n = len(y_true)

        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            T = true_groups[t]
            C = pred_groups[p]
            inter = len(T & C)

            P_sum += inter / len(C) if len(C) > 0 else 0.0
            R_sum += inter / len(T) if len(T) > 0 else 0.0

        P = P_sum / n
        R = R_sum / n
        F1 = (2 * P * R / (P + R)) if (P + R) > 0 else 0.0

        return P, R, F1

    b3_p, b3_r, b3_f1 = b3_precision_recall_f1(y_true, y_pred)

    
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    hungarian_acc = cm[row_ind, col_ind].sum() / cm.sum()

    
    print(f"Purity : {overall_purity:.4f}")
    print(f"ACC    : {hungarian_acc:.4f}")
    
    
    print(f"B3-F1  : {b3_f1:.4f}")
    
    
    print(f"V-F1   : {v_f1:.4f}")
    print(f"ARI    : {ari:.4f}")

    return {
        "Purity": overall_purity,
        "ARI": ari,
        "V-hom": hom,
        "V-com": com,
        "V-F1": v_f1,
        "B3-P": b3_p,
        "B3-R": b3_r,
        "B3-F1": b3_f1,
        "ACC": hungarian_acc,
    }

In [6]:


metrics = evaluate_clustering(
    df=df_train,
    pred_col="cluster_unknown_k",
    gold_col="relation"
)

metrics

Purity : 0.6054
ACC    : 0.4216
B3-F1  : 0.3857
V-F1   : 0.6582
ARI    : 0.3558


{'Purity': 0.6054054054054054,
 'ARI': 0.3557928143476812,
 'V-hom': 0.713509873064178,
 'V-com': 0.6108389910221272,
 'V-F1': 0.6581946234348803,
 'B3-P': 0.5057820113904926,
 'B3-R': 0.31176216216216196,
 'B3-F1': 0.3857496599079401,
 'ACC': 0.42162162162162165}

# Test set Evaluation

In [10]:
test_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/RE-TACRED/Test/RE_TACRED_test_phrase.csv')
print(test_df.shape)
test_df.head(2)

(740, 14)


,split,sample_id,text,token,subject,subj_start,subj_end,subj_type,object,obj_start,obj_end,obj_type,relation,gpt4_phrase
0,test,098f6a9e36dad0a62b98,The general assembly of the Organisation of As...,"['The', 'general', 'assembly', 'of', 'the', 'O...",Organisation of Asia-Pacific News Agencies,5,9,ORGANIZATION,OANA,11,11,ORGANIZATION,org:alternate_names,abbreviation_expansion
1,test,098f665fb9e7a0eac10f,The general assembly of the Organisation of As...,"['The', 'general', 'assembly', 'of', 'the', 'O...",OANA,11,11,ORGANIZATION,OANA,33,33,ORGANIZATION,org:alternate_names,Organization-GoverningBodyRelationship


In [11]:



if "gpt4_phrase" not in test_df.columns:
    raise ValueError("test_df에 'gpt4_phrase' 컬럼이 없습니다.")

if "relation" not in test_df.columns:
    raise ValueError("test_df에 gold label 컬럼 'relation'이 없습니다.")

test_texts = test_df["gpt4_phrase"].fillna("").astype(str).tolist()

print("num test instances:", len(test_texts))

test_embeddings = model.encode(
    test_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("test_embeddings shape:", test_embeddings.shape)

test_df["cluster_unknown_k"] = kmeans.predict(test_embeddings)

print(test_df[[
    "relation",
    "gpt4_phrase",
    "cluster_unknown_k"
]].head(10))

num test instances: 740


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

test_embeddings shape: (740, 384)
              relation                             gpt4_phrase  \
0  org:alternate_names                  abbreviation_expansion   
1  org:alternate_names  Organization-GoverningBodyRelationship   
2  org:alternate_names                 organization-has-office   
3  org:alternate_names                                   alias   
4  org:alternate_names                      Acronym_Definition   
5  org:alternate_names                                   alias   
6  org:alternate_names                                   alias   
7  org:alternate_names                event_organizer_of_event   
8  org:alternate_names                  abbreviation_expansion   
9  org:alternate_names                                alias_of   

   cluster_unknown_k  
0                 40  
1                 70  
2                 45  
3                 12  
4                 40  
5                 12  
6                 12  
7                 16  
8                 40  
9        

In [12]:
metrics_kmeans_test = evaluate_clustering(
    df=test_df,
    pred_col="cluster_unknown_k",
    gold_col="relation"
)

metrics_kmeans_test

Purity : 0.6311
ACC    : 0.4568
B3-F1  : 0.4253
V-F1   : 0.7053
ARI    : 0.3735


{'Purity': 0.6310810810810811,
 'ARI': 0.3734913118342177,
 'V-hom': 0.7599738681847226,
 'V-com': 0.658031422017461,
 'V-F1': 0.7053382503338892,
 'B3-P': 0.5460188460188472,
 'B3-R': 0.3482432432432435,
 'B3-F1': 0.4252609522258481,
 'ACC': 0.45675675675675675}